# 🏔️ Quantum Ascent — Basecamp 3: Entanglement

<div style="border-left:5px solid #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:4px">
<b>📋 Mission briefing.</b> So far every qubit has lived its own life. Now you'll <b>link two of them</b> so tightly that they stop being two separate coins and become one shared object — measure one and you instantly know the other. This is <b>entanglement</b>, the single most famous idea in quantum computing, and the engine behind almost everything ahead.<br><br>You'll meet the <b>CNOT</b> gate (the move that does the linking), build your first <b>Bell pair</b>, and see correlations no pair of ordinary coins could ever produce.<br><br><i>We'll also carefully dismantle the biggest myth here — no, entanglement does <b>not</b> let you send messages faster than light. You'll see exactly why, with your own hands.</i>
</div>

**By the end of this basecamp you can:**
- Describe why two qubits need **four** amplitudes ($|00\rangle, |01\rangle, |10\rangle, |11\rangle$) — the state space grows as $2^n$
- Use the **CNOT** gate to link qubits, and build the **Bell pair** $(|00\rangle+|11\rangle)/\sqrt{2}$
- Explain **entanglement** as a state that *cannot* be split into 'qubit A does this, qubit B does that'
- Bust the myth: measuring one entangled qubit reveals — but does **not signal** — its partner (no faster-than-light messaging)

*Estimated time: 55 min · Best experienced with the [course website](https://quantum-ascent-77617.web.app) open in another tab.*

In [ ]:
# ✅ Setup — run me first! (works locally and on Google Colab)
import importlib.util, os, sys, subprocess, urllib.request

def _ensure_q2q():
    for rel in (".", "..", "../.."):
        if os.path.isdir(os.path.join(rel, "q2q")):
            sys.path.insert(0, os.path.abspath(rel))
            break
    if importlib.util.find_spec("q2q") is not None:
        return
    # On Colab: install the pinned SDKs and fetch the course helpers
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "qiskit==2.3.1", "qiskit-aer==0.17.2", "pylatexenc"], check=False)
    base = ("https://raw.githubusercontent.com/sahgyan9/quantum-ascent/"
            "main/notebooks/q2q/")
    os.makedirs("q2q", exist_ok=True)
    for fname in ("__init__.py", "checkers.py", "widgets.py", "oracles.py",
                  "latex_macros.py", "targets.py", "quiz.py", "progress.py"):
        urllib.request.urlretrieve(base + fname, os.path.join("q2q", fname))
    sys.path.insert(0, os.path.abspath("."))

_ensure_q2q()
from q2q import checkers, quiz, targets, progress
from q2q.widgets import show_widget
print("Setup complete — you're ready to climb. 🏔️")

## 3.1 Two qubits: the world gets bigger

Everything so far lived on a *single* qubit — one arrow, two amplitudes, one probability split. The moment we add a **second** qubit, something important happens to the bookkeeping.

One qubit had two possible outcomes: $0$ or $1$. **Two** qubits have **four**: $00, 01, 10, 11$. So a two-qubit state needs **four amplitudes**, one for each of those outcomes:

$$|\psi\rangle = a\,|00\rangle + b\,|01\rangle + c\,|10\rangle + d\,|11\rangle$$

(As always, the probabilities are the squares — $|a|^2 + |b|^2 + |c|^2 + |d|^2 = 1$.)

That's the pattern: **$n$ qubits need $2^n$ amplitudes.** Ten qubits already need 1024 numbers; three hundred qubits need more numbers than there are atoms in the visible universe. *That* explosion is where quantum computers get their room to work — and it starts right here, with going from 2 numbers to 4.

> 🧭 **Don't panic about the four terms.** You won't be juggling them by hand. We'll *watch* them in the Explorer first, then let Qiskit track them for us. The point of this section is just one idea: **two qubits share one bigger state.**

<div style="border-left:5px solid #0e7490;background:#eaf4f6;color:#0c363f;padding:12px 16px;border-radius:4px">
<b>🔭 Exercise 1.</b> Open the <b>Entanglement Explorer</b> below. It shows two qubits — <b>Qubit A</b> and <b>Qubit B</b> — and lets you measure them together. <br>1️⃣ Leave the link on <b>Independent</b> and hit <b>Measure ×100</b>. All four pairs (00, 01, 10, 11) show up, each about 25%. This is two ordinary, unconnected coins. <br>2️⃣ Watch the little <b>ink goal marks</b> on each bar — that's the exact probability the bar climbs toward (the Born rule from Basecamp 1, now for a pair). <br>3️⃣ Keep this open; in a moment we'll change the link and something strange will happen.
</div>

In [ ]:
show_widget("entanglement-explorer")

### When two qubits are really just two separate stories

The Independent link you just played with has a special property: it **factors**. "Qubit A is a fair coin" *and*, separately, "Qubit B is a fair coin." You can tell each qubit's story on its own, and stapling them together describes the whole. A state like that is called a **product state** — it's genuinely two objects that happen to sit next to each other.

Most two-qubit states are like this. But not all of them. Let's go find one that *isn't*.

<div style="border-left:5px solid #0e7490;background:#eaf4f6;color:#0c363f;padding:12px 16px;border-radius:4px">
<b>🔭 Exercise 2.</b> Back in the Explorer, switch the link to <b>Correlated (a Bell pair)</b> and hit <b>Measure ×100</b>. <br>1️⃣ Two of the bars — <b>01</b> and <b>10</b> — stay flat at zero. Only <b>00</b> and <b>11</b> ever appear. <br>2️⃣ Look at the <b>“How often the two coins agreed”</b> readout: 100%. The coins <i>always</i> land on the same face. <br>3️⃣ Yet each coin <i>alone</i> is still a fair 50/50 — cover one and the other is pure chance. It's the <b>link</b> that's certain, not either coin. Sit with how odd that is, then answer the quick check.
</div>

In [ ]:
# Quick check — with the Correlated (Bell) link, which outcomes appear?
quiz.ask("m3-bell-outcomes")

### Naming it: entanglement

That correlated state is famous enough to have a name — the **Bell state** $\Phi^+$:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

Read it out loud: "an equal mix of *both zero* and *both one* — and nothing else." There's no $|01\rangle$ term and no $|10\rangle$ term, so a mismatch is simply impossible.

Here's the crucial part. Try to split $|\Phi^+\rangle$ into "Qubit A is doing ___ *and* Qubit B is doing ___." **You can't.** No description of A by itself, combined with any description of B by itself, ever reproduces this state. The only honest description is of the **pair, as one object**.

That un-splittability *is* the definition:

> 🔗 **Entanglement:** a state of two (or more) qubits that **cannot** be written as one qubit's state *and* another's. The qubits no longer have separate identities — only the whole does.

A product state = two stories. An entangled state = **one story you can't tear in half.**

## 3.2 Building a Bell pair in Qiskit

How do you actually *link* two qubits? With one new gate: the **CNOT** (controlled-NOT), written `cx` in Qiskit.

CNOT is a two-qubit move with a **control** and a **target**:

> 🎯 **CNOT rule:** look at the control qubit. If it's $|1\rangle$, flip the target. If it's $|0\rangle$, do nothing.

On its own that's just a conditional bit-flip — very classical. The magic appears when the control is in *superposition*: then "flip / don't flip" happens for both possibilities at once, and the two qubits get braided together.

Same lab as before — `QuantumCircuit`, `AerSimulator`, and NumPy. Run the setup:

In [ ]:
# Our familiar tools.
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

### Worked example: CNOT as a plain conditional flip

Let's watch the CNOT rule with **no** superposition first, so it's totally concrete. We'll force the control (qubit 0) to $|1\rangle$ with an $X$, then apply `cx(0, 1)` — control 0, target 1. Since the control is 1, the target should flip. Predict the bitstring before you run (remember Basecamp 2: **qubit 0 is the rightmost character**):

In [ ]:
# Control = qubit 0, target = qubit 1. Force the control to 1, then CNOT.
qc_cnot = QuantumCircuit(2)
qc_cnot.x(0)          # control becomes |1>
qc_cnot.cx(0, 1)      # control is 1 -> flip the target (qubit 1)
qc_cnot.measure_all()

# Both qubits should now read 1 -> '11' every shot.
counts_cnot = checkers.run_and_tally(qc_cnot, shots=200)
checkers.check_counts_close(counts_cnot, {"11": 1.0})

Control was $1$, so the target flipped — both qubits ended up $1$, giving `'11'`. A CNOT with a *definite* control is nothing exotic. Now let's make the control a **spin** instead.

### The recipe for entanglement

Put the control qubit into a fair superposition with $H$ **first**, *then* CNOT:

$$|0\rangle|0\rangle \;\xrightarrow{\;H\text{ on qubit }0\;}\; \frac{|0\rangle+|1\rangle}{\sqrt{2}}|0\rangle \;\xrightarrow{\;\text{CNOT}\;}\; \frac{|00\rangle+|11\rangle}{\sqrt{2}}$$

Watch what CNOT does to the two pieces of the superposition: the $|0\rangle$-control part leaves the target alone (stays $|00\rangle$), while the $|1\rangle$-control part flips the target (becomes $|11\rangle$). The "flip / don't flip" got tied to the control's coin-flip — and that knot is the entanglement. Two gates, $H$ then CNOT, is the entire Bell-pair recipe. Your turn to build it.

<div style="border-left:5px solid #b45309;background:#fdf3e7;color:#5c2d0c;padding:12px 16px;border-radius:4px">
<b>⛏️ Task 1.</b> Build the Bell pair <code>qc_bell</code>. Starting from a fresh 2-qubit circuit, apply <b>H to qubit 0</b>, then <b>CNOT with control 0, target 1</b> (<code>.cx(0, 1)</code>). <br><i>Don't measure it</i> — this time we check the actual <b>state vector</b> to confirm it's exactly $(|00\rangle+|11\rangle)/\sqrt{2}$: two amplitudes of $1/\sqrt{2}$ on $|00\rangle$ and $|11\rangle$, and zero on the mismatched outcomes.
</div>

In [ ]:
# Build the Bell pair: H on qubit 0, then CNOT (control 0 -> target 1).
qc_bell = QuantumCircuit(2)
# YOUR CODE HERE

# Nothing to change below — we compare your state to the Phi+ Bell target.
checkers.check_statevector(qc_bell, targets.M3_BELL)

#### 🔬 Analysis

You just built genuine entanglement in two lines. The state
$(|00\rangle+|11\rangle)/\sqrt{2}$ has equal amplitude on "both 0" and "both 1", and
**exactly zero** on the mismatched $|01\rangle$ and $|10\rangle$ — which is why the
Explorer's 01 and 10 bars stayed flat. Notice the checker forgives global phase
but nothing else: your two amplitudes have to be the same, and the cross terms
have to vanish. If they do, the qubits are locked together.

In [ ]:
# Quick check — you measure qubit A of a Bell pair and get 0. What is qubit B?
quiz.ask("m3-measure-partner")

## 3.3 Correlation, the other Bell pair, and a myth to bury

The Bell pair $\Phi^+$ makes the qubits **agree**. With a tiny tweak we can make them do the opposite — **always disagree**. Take the Bell recipe and flip one qubit at the end with an $X$: now the only outcomes are $|01\rangle$ and $|10\rangle$. Same entanglement, mirror-image correlation (this is the Explorer's **Anti-correlated** link, the Bell state $\Psi^+$).

Before you build it, let's name the elephant in the room.

> 🚫 **Myth:** "Entangled qubits communicate instantly, so you can send messages faster than light."
>
> ✅ **Reality:** You cannot. Look *only* at Qubit A across many Bell pairs and you see a boring, perfectly random 50/50 stream of 0s and 1s — measuring B does nothing you could ever detect on A. The correlation is real, but it only *appears* when someone later lays the two lists of results **side by side and compares them**. Comparing lists needs an ordinary (slower-than-light) message. So entanglement gives you shared randomness that's *linked*, never a telephone. No signal travels; nothing arrives sooner than light.

Why does the "spooky action" story feel so tempting, then? Because measuring A really does pin down B's value. But "learning something about B" isn't "sending something to B." Opening one of two envelopes that were sealed together tells you what's in the other instantly, too — no magic, and definitely no message. Entanglement is stranger than sealed envelopes (the correlations can beat *any* pre-arranged classical scheme — that's the deep part, for a later climb), yet it still refuses to carry a signal.

In [ ]:
# Quick check — which state is genuinely entangled (can't be split in two)?
quiz.ask("m3-entangled-or-not")

<div style="border-left:5px solid #b45309;background:#fdf3e7;color:#5c2d0c;padding:12px 16px;border-radius:4px">
<b>⛏️ Task 2.</b> Build <code>qc_anti</code>, the <b>anti-correlated</b> pair. Start exactly like the Bell pair — <b>H on qubit 0</b>, then <b>CNOT (0 → 1)</b> — then add <b>one</b> more move: an <b>X on qubit 1</b> to flip it. The measurement is already wired for you. <br><i>Predict first:</i> flipping one half of a 'they always agree' pair should make them 'always disagree' — so only <b>'01'</b> and <b>'10'</b> should ever appear, about half each.
</div>

In [ ]:
# The anti-correlated pair: Bell recipe, then flip qubit 1 so they disagree.
qc_anti = QuantumCircuit(2)
# YOUR CODE HERE

# Nothing to change below — measure, then check the outcomes are 01 / 10 only.
qc_anti.measure_all()
counts_anti = checkers.run_and_tally(qc_anti, shots=512)
checkers.check_counts_close(counts_anti, {"01": 0.5, "10": 0.5})

#### 🔬 Analysis

Perfect anti-correlation: only `'01'` and `'10'`, so the two
qubits land on opposite faces every single time — yet each one alone is still a
fair 50/50. By flipping just one qubit you turned "always agree" into "always
disagree" without breaking the entanglement. $\Phi^+$ and $\Psi^+$ are two of the
four **Bell states**, the fundamental alphabet of two-qubit entanglement — and
they're the raw material for teleportation, superdense coding, and error
correction further up the mountain.

<div style="border:1px dashed #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:8px">
<b>🎨 Make it yours — analogy time.</b> Everyone's brain hooks onto different things. Copy the prompt below into <i>your</i> favorite AI (ChatGPT, Claude, Gemini) and get <b>entanglement and Bell pairs (without the faster-than-light myth)</b> explained through <i>your</i> world. The precise definition is baked into the prompt, so the AI can't drift into pop-science myths. Want more control? Use the <a href="https://quantum-ascent-77617.web.app/analogy-studio.html">Analogy Studio</a>.

<pre style="white-space:pre-wrap;background:#ffffff;border:1px solid #cfe6d8;color:#0f3d22;border-radius:6px;padding:10px">I'm learning quantum computing. Explain quantum entanglement using an analogy from MY background: [YOUR HOBBY/FIELD HERE].

Ground rules — your analogy MUST respect these facts:
1) Two entangled qubits share ONE joint state that cannot be described as 'qubit A does X and qubit B does Y' separately.
2) The Bell pair (|00>+|11>)/sqrt(2) always gives matching results (00 or 11), each 50%; each qubit measured alone is a fair 50/50.
3) Measuring one qubit reveals its partner's value but does NOT send any signal — you can't communicate faster than light. The correlation is only visible when the two result lists are later compared.
4) Do NOT say the qubits 'communicate' or 'influence each other faster than light'. Avoid 'spooky action' as a literal mechanism.

End by telling me where the analogy breaks down.</pre>
</div>

## 🎓 Log your climb — claim your completion code

You built real entanglement: a Bell pair whose qubits always agree, and its
anti-correlated twin whose qubits always disagree. Let's bank it! Run the cell
below — it re-checks your **Task 1** (`qc_bell`) and **Task 2** (`qc_anti`) right
here in this kernel and, if they pass, prints your personal **completion code**.

Copy that code into the **“Log your notebook”** box on the
[Basecamp 3 page](https://quantum-ascent-77617.web.app/module.html?id=03#claim)
to light up this camp on your Ascent map and claim the **🏅 Entangled** badge. 🏔️

In [ ]:
progress.claim_basecamp_3(qc_bell, qc_anti)

<div style="border-left:5px solid #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:4px">
<b>🚩 Basecamp 3 reached!</b> You linked two qubits into one shared state. You met the **CNOT** gate, built the Bell pair $(|00\rangle+|11\rangle)/\sqrt{2}$, saw that entanglement is a state you *cannot* split in two, made an anti-correlated pair with a single extra flip, and buried the faster-than-light myth for good. This is the beating heart of quantum computing — and you just built it yourself.
</div>

**Next steps:**
1. 🧠 Take the [Basecamp 3 quiz](https://quantum-ascent-77617.web.app/module.html?id=03#quiz) on the website to earn your XP and badge.
2. 🧗 Continue the ascent: **Basecamp 4: Hamiltonians & Energy — how a quantum computer measures the 'cost' of a state**.

---
*Stuck on a task? Compare with the worked solutions: [`solutions/03_entanglement_and_multiqubit_solutions.ipynb`](solutions/03_entanglement_and_multiqubit_solutions.ipynb). Try honestly first — the struggle is where the learning happens.*